<a href="https://colab.research.google.com/github/Priyanka-c0de/ITA0606-ML/blob/main/ML_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Session 1: FIND-S Algorithm
import csv
import urllib.request

def find_s_algorithm(url_or_file):
    # If it's a URL, download and read it
    if url_or_file.startswith('http://') or url_or_file.startswith('https://'):
        with urllib.request.urlopen(url_or_file) as response:
            lines = [line.decode('utf-8') for line in response.readlines()]
            reader = list(csv.reader(lines))
    else:
        with open(url_or_file, 'r') as file:
            reader = list(csv.reader(file))

    header = reader[0]
    data = reader[1:]

    # Initialize hypothesis with the most specific hypothesis: ['0', '0', ...]
    num_attributes = len(data[0]) - 1
    hypothesis = ['0'] * num_attributes

    print("Initial Hypothesis:", hypothesis)
    print("-" * 50)

    for i, row in enumerate(data):
        features = row[:-1]
        target = row[-1]

        # Consider only positive training instances
        if target.strip().lower() in ['yes', '1', 'positive']:
            if hypothesis == ['0'] * num_attributes:
                # First positive example initializes the hypothesis
                hypothesis = features.copy()
            else:
                for j in range(num_attributes):
                    if hypothesis[j] != features[j]:
                        hypothesis[j] = '?'
            print(f"Instance {i + 1} ({target}): {features}")
            print(f"Hypothesis after instance {i + 1}: {hypothesis}\n")
        else:
            print(f"Instance {i + 1} ({target}): {features} (Ignored)\n")

    print("-" * 50)
    print("Most Specific Hypothesis Found:")
    print(hypothesis)

# Run Find-S using your GitHub data
github_training_data_url = 'https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/training_data.csv'
find_s_algorithm(github_training_data_url)

Initial Hypothesis: ['0', '0', '0', '0', '0', '0']
--------------------------------------------------
Instance 1 (Yes): ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Hypothesis after instance 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']

Instance 2 (Yes): ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same']
Hypothesis after instance 2: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']

Instance 3 (No): ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] (Ignored)

Instance 4 (Yes): ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change']
Hypothesis after instance 4: ['Sunny', 'Warm', '?', 'Strong', '?', '?']

--------------------------------------------------
Most Specific Hypothesis Found:
['Sunny', 'Warm', '?', 'Strong', '?', '?']


In [ ]:
# Session 2: Candidate-Elimination Algorithm
import csv
import urllib.request

def candidate_elimination(url_or_file):
    # If it's a URL, download and read it
    if url_or_file.startswith('http://') or url_or_file.startswith('https://'):
        with urllib.request.urlopen(url_or_file) as response:
            lines = [line.decode('utf-8') for line in response.readlines()]
            reader = list(csv.reader(lines))
    else:
        with open(url_or_file, 'r') as file:
            reader = list(csv.reader(file))

    header = reader[0]
    data = reader[1:]

    num_attributes = len(data[0]) - 1

    # Initialize Specific and General Boundaries
    specific_h = ['0'] * num_attributes
    general_h = [['?' for _ in range(num_attributes)]]

    # Initialize S with the first positive example
    for row in data:
        if row[-1].strip().lower() in ['yes', '1', 'positive']:
            specific_h = row[:-1].copy()
            break

    print("Initial Specific Boundary (S):", specific_h)
    print("Initial General Boundary (G):", general_h)
    print("-" * 60)

    for idx, row in enumerate(data):
        features = row[:-1]
        label = row[-1].strip().lower()

        if label in ['yes', '1', 'positive']:
            # Positive Example
            # Remove any hypothesis from G that is inconsistent with features
            general_h = [
                g for g in general_h
                if all(g[i] == '?' or g[i] == features[i] for i in range(num_attributes))
            ]

            # Generalize S to cover features
            for i in range(num_attributes):
                if specific_h[i] != features[i]:
                    specific_h[i] = '?'

        else:
            # Negative Example
            # Specialize G to exclude features while remaining consistent with S
            new_general_h = []
            for g in general_h:
                if not all(g[i] == '?' or g[i] == features[i] for i in range(num_attributes)):
                    new_general_h.append(g)
                else:
                    for i in range(num_attributes):
                        if g[i] == '?' and specific_h[i] != '?' and specific_h[i] != features[i]:
                            spec = g.copy()
                            spec[i] = specific_h[i]
                            new_general_h.append(spec)

            # Filter duplicates and redundant hypotheses
            general_h = [
                h for i, h in enumerate(new_general_h)
                if h not in new_general_h[:i]
            ]

        print(f"Step {idx + 1} (Instance: {features} -> {row[-1]}):")
        print("  S:", specific_h)
        print("  G:", general_h)
        print("-" * 60)

    print("Final Version Space:")
    print("Specific Boundary (S):", specific_h)
    print("General Boundary (G):", general_h)

# Run Candidate-Elimination using your GitHub data
github_training_data_url = 'https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/training_data.csv'
candidate_elimination(github_training_data_url)

Initial Specific Boundary (S): ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Initial General Boundary (G): [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 1 (Instance: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Yes):
  S: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
  G: [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 2 (Instance: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Yes):
  S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G: [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 3 (Instance: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> No):
  S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]
------------------------------------------------------------
Step 4 (Insta

In [ ]:
# Session 3: Decision Tree using ID3 Algorithm
import pandas as pd
import numpy as np

# 1. Calculate Entropy of a dataset
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = 0
    for i in range(len(elements)):
        probability = counts[i] / np.sum(counts)
        entropy_val -= probability * np.log2(probability)
    return entropy_val

# 2. Calculate Information Gain for a feature
def information_gain(data, split_attribute_name, target_attribute_name="PlayTennis"):
    total_entropy = entropy(data[target_attribute_name])

    vals, counts = np.unique(data[split_attribute_name], return_counts=True)
    weighted_entropy = 0

    for i in range(len(vals)):
        sub_data = data[data[split_attribute_name] == vals[i]]
        prob = counts[i] / np.sum(counts)
        weighted_entropy += prob * entropy(sub_data[target_attribute_name])

    info_gain = total_entropy - weighted_entropy
    return info_gain

# 3. ID3 Algorithm Recursive Function
def id3(data, original_data, features, target_attribute_name="PlayTennis", parent_node_class=None):
    # Case 1: If target values are homogeneous (all 'Yes' or all 'No')
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # Case 2: If dataset is empty, return target feature value in original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[
            np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])
        ]

    # Case 3: If feature set is empty, return parent node class
    elif len(features) == 0:
        return parent_node_class

    # Standard Case: Build tree recursively
    else:
        # Determine majority class for current node
        counts = np.unique(data[target_attribute_name], return_counts=True)[1]
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(counts)]

        # Select feature with maximum Information Gain
        item_values = [information_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create tree root structure
        tree = {best_feature: {}}

        # Remove best feature from feature list
        features = [i for i in features if i != best_feature]

        # Recursively grow tree for each branch/value
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, original_data, features, target_attribute_name, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

# 4. Predict function for new unseen sample
def predict(query, tree, default='Yes'):
    for key in list(query.keys()):
        if key in list(tree.keys()):
            try:
                result = tree[key][query[key]]
            except KeyError:
                return default

            result = tree[key][query[key]]
            if isinstance(result, dict):
                return predict(query, result)
            else:
                return result

# --- Main Execution ---
if __name__ == "__main__":
    # GitHub Raw URL for play_tennis.csv
    url = "https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/play_tennis.csv"

    # Load dataset directly from URL
    df = pd.read_csv(url)

    # Clean data (Remove 'Day' or similar index columns if present)
    if 'Day' in df.columns:
        df = df.drop(columns=['Day'])
    if 'Humidity' in df.columns:
        df = df[df['Humidity'] != '???']

    # Features & Target Column
    features = [col for col in df.columns if col != 'PlayTennis']

    # Build Decision Tree
    decision_tree = id3(df, df, features)

    print("Constructed Decision Tree Structure:")
    import pprint
    pprint.pprint(decision_tree)
    print("-" * 50)

    # Predict a new sample
    new_sample = {
        'Outlook': 'Sunny',
        'Temperature': 'Cool',
        'Humidity': 'High',
        'Wind': 'Strong'
    }

    prediction = predict(new_sample, decision_tree)
    print("New Sample:", new_sample)
    print("Classification Result for New Sample (PlayTennis):", prediction)

Constructed Decision Tree Structure:
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Light': 'Yes', 'Strong': 'No'}},
             'Sunny': {'Temperature': {'Cool': 'Yes',
                                       'Hot': 'No',
                                       'Mild': 'Yes'}}}}
--------------------------------------------------
New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Classification Result for New Sample (PlayTennis): Yes


In [ ]:
# Session 4: Artificial Neural Network with Backpropagation

import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([
    [0],
    [1],
    [1],
    [0]
])


np.random.seed(42)


epochs = 10000
learning_rate = 0.1
input_size = 2
hidden_size = 3
output_size = 1


wh = np.random.uniform(size=(input_size, hidden_size))
bh = np.random.uniform(size=(1, hidden_size))


wout = np.random.uniform(size=(hidden_size, output_size))
bout = np.random.uniform(size=(1, output_size))

print("--- Training Progress ---")

for epoch in range(epochs):

    hidden_layer_input = np.dot(X, wh) + bh
    hidden_layer_activations = sigmoid(hidden_layer_input)

    output_layer_input = np.dot(hidden_layer_activations, wout) + bout
    predicted_output = sigmoid(output_layer_input)

    error = y - predicted_output
    d_predicted_output = error * sigmoid_derivative(predicted_output)


    error_hidden_layer = d_predicted_output.dot(wout.T)
    d_hidden_layer = error_hidden_layer * sigmoid_derivative(hidden_layer_activations)


    wout += hidden_layer_activations.T.dot(d_predicted_output) * learning_rate
    bout += np.sum(d_predicted_output, axis=0, keepdims=True) * learning_rate

    wh += X.T.dot(d_hidden_layer) * learning_rate
    bh += np.sum(d_hidden_layer, axis=0, keepdims=True) * learning_rate


    if (epoch + 1) % 2000 == 0 or epoch == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch + 1}/{epochs} - Loss (MSE): {loss:.6f}")

print("-" * 50)
print("Training Completed.\n")


print("--- Testing Results ---")
print("Input 1 | Input 2 | Target Output | Predicted Output | Classification")
print("-" * 68)
for i in range(len(X)):

    hidden_act = sigmoid(np.dot(X[i], wh) + bh)
    pred = sigmoid(np.dot(hidden_act, wout) + bout)[0][0]
    binary_class = 1 if pred > 0.5 else 0
    print(f"   {X[i][0]}    |    {X[i][1]}    |       {y[i][0]}       |     {pred:.4f}     |       {binary_class}")

--- Training Progress ---
Epoch 1/10000 - Loss (MSE): 0.394174
Epoch 2000/10000 - Loss (MSE): 0.221589
Epoch 4000/10000 - Loss (MSE): 0.142439
Epoch 6000/10000 - Loss (MSE): 0.128289
Epoch 8000/10000 - Loss (MSE): 0.018662
Epoch 10000/10000 - Loss (MSE): 0.007339
--------------------------------------------------
Training Completed.

--- Testing Results ---
Input 1 | Input 2 | Target Output | Predicted Output | Classification
--------------------------------------------------------------------
   0    |    0    |       0       |     0.0501     |       0
   0    |    1    |       1       |     0.9208     |       1
   1    |    0    |       1       |     0.9036     |       1
   1    |    1    |       0       |     0.1061     |       0


In [ ]:
# Session 5 : Naïve Bayes Algorithm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


def generate_dataset(filename="dataset_session5.csv"):
    np.random.seed(42)
    n_samples = 200

    age = np.random.randint(18, 65, size=n_samples)
    salary = np.random.randint(20000, 120000, size=n_samples)
    credit_score = np.random.randint(300, 850, size=n_samples)

    purchased = ((age * 0.4 + (salary / 2000) * 0.6) > 35).astype(int)

    df = pd.DataFrame({
        'Age': age,
        'EstimatedSalary': salary,
        'CreditScore': credit_score,
        'Purchased': purchased
    })


    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_naive_bayes(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['Age', 'EstimatedSalary', 'CreditScore']]
    y = df['Purchased']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    nb_model = GaussianNB()

    nb_model.fit(X_train, y_train)

    y_pred = nb_model.predict(X_test)


    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)

    print("\nConfusion Matrix:")
    print(cm)
    print(f"\nMatrix Structure:")
    print(f"  [ True Negatives (TN): {cm[0][0]} | False Positives (FP): {cm[0][1]} ]")
    print(f"  [ False Negatives (FN): {cm[1][0]} | True Positives (TP):  {cm[1][1]} ]")

    print(f"\nAccuracy Score: {acc * 100:.2f}%")

    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("dataset_session5.csv")
    run_naive_bayes(dataset_file)

Dataset successfully created and saved as 'dataset_session5.csv'.

--- Dataset Preview ---
   Age  EstimatedSalary  CreditScore  Purchased
0   56            72733          335          1
1   46            85318          319          1
2   32           109474          620          1
3   60            43664          811          1
4   25            87172          699          1 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[18  6]
 [ 0 16]]

Matrix Structure:
  [ True Negatives (TN): 18 | False Positives (FP): 6 ]
  [ False Negatives (FN): 0 | True Positives (TP):  16 ]

Accuracy Score: 85.00%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.75      0.86        24
           1       0.73      1.00      0.84        16

    accuracy                           0.85        40
   macro avg       0.86      0.88      0.85    

In [ ]:
# Session 6: House Price Prediction using Linear Regression
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


def generate_dataset(filename="house_prices.csv"):
    np.random.seed(42)
    n_samples = 250

    square_feet = np.random.randint(600, 3500, size=n_samples)
    bedrooms = np.random.randint(1, 6, size=n_samples)
    age = np.random.randint(1, 30, size=n_samples)
    distance_to_center = np.random.uniform(1.0, 25.0, size=n_samples).round(1)

    base_price = (
        square_feet * 150 +
        bedrooms * 10000 -
        age * 1200 -
        distance_to_center * 2500 +
        30000
    )
    noise = np.random.normal(0, 15000, size=n_samples)
    price = (base_price + noise).round(2)

    df = pd.DataFrame({
        'SquareFeet': square_feet,
        'Bedrooms': bedrooms,
        'Age': age,
        'DistanceToCenter': distance_to_center,
        'Price': price
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename


def run_house_price_prediction(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['SquareFeet', 'Bedrooms', 'Age', 'DistanceToCenter']]
    y = df['Price']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    model = LinearRegression()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print(f"Mean Absolute Error (MAE) : ${mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R² Score (Accuracy)       : {r2 * 100:.2f}%\n")

    print("--- Feature Coefficients ---")
    for feature, coef in zip(X.columns, model.coef_):
        print(f"  {feature:18s}: ${coef:,.2f}")
    print(f"  Intercept         : ${model.intercept_:,.2f}\n")

    sample_house = pd.DataFrame({
        'SquareFeet': [2000],
        'Bedrooms': [3],
        'Age': [5],
        'DistanceToCenter': [8.5]
    })
    predicted_price = model.predict(sample_house)[0]

    print("--- Sample Prediction ---")
    print(f"Features: 2000 sqft, 3 Bed, 5 years old, 8.5 km from center")
    print(f"Predicted Price: ${predicted_price:,.2f}")

if __name__ == "__main__":
    dataset_file = generate_dataset("house_prices.csv")
    run_house_price_prediction(dataset_file)

Dataset successfully created and saved as 'house_prices.csv'.

--- Dataset Preview ---
   SquareFeet  Bedrooms  Age  DistanceToCenter      Price
0        1460         2   24              19.0  181008.61
1        1894         2    4              10.4  270352.28
2        1730         3   25              20.9  238077.34
3        1695         4   21              14.7  299885.34
4        2238         2   29               2.5  329370.71 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------
Mean Absolute Error (MAE) : $14,346.19
Root Mean Squared Error (RMSE): $17,118.90
R² Score (Accuracy)       : 97.89%

--- Feature Coefficients ---
  SquareFeet        : $149.22
  Bedrooms          : $10,274.56
  Age               : $-1,019.83
  DistanceToCenter  : $-2,345.91
  Intercept         : $27,573.22

--- Sample Prediction ---
Features: 2000 sqft, 3 Bed, 5 years old, 8.5 km from center
Predicted Price: $331,801.17


In [ ]:
# Session 7: Iris Flower Classification using Naïve Bayes Classifier

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


def generate_dataset(filename="iris_dataset.csv"):
    iris = load_iris()

    df = pd.DataFrame(data=iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
    df['species'] = iris.target_names[iris.target]

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename


def run_iris_classification(csv_file):

    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
    y = df['species']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    nb_model = GaussianNB()

    nb_model.fit(X_train, y_train)

    y_pred = nb_model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)

    print("\nConfusion Matrix:")
    print(cm)

    print(f"\nAccuracy Score: {acc * 100:.2f}%")

    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("iris_dataset.csv")
    run_iris_classification(dataset_file)

Dataset successfully created and saved as 'iris_dataset.csv'.

--- Dataset Preview ---
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Accuracy Score: 100.00%

Detailed Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg  

In [ ]:
# Session 8: Implementation of Naïve Bayes Classification for Bank Loan Prediction

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


def generate_dataset(filename="bank_loan_dataset.csv"):
    np.random.seed(42)
    n_samples = 300

    # Features: Income (in k$), Credit Score, Debt-to-Income Ratio (%), Loan Amount (in k$)
    income = np.random.randint(20, 150, size=n_samples)
    credit_score = np.random.randint(300, 850, size=n_samples)
    dti_ratio = np.random.uniform(5.0, 50.0, size=n_samples).round(1)
    loan_amount = np.random.randint(5, 50, size=n_samples)

    approval_score = (income * 0.4) + (credit_score * 0.08) - (dti_ratio * 0.8) - (loan_amount * 0.5)
    loan_approved = (approval_score > 45).astype(int)

    df = pd.DataFrame({
        'Income': income,
        'CreditScore': credit_score,
        'DTIRatio': dti_ratio,
        'LoanAmount': loan_amount,
        'LoanApproved': loan_approved
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename


def run_bank_loan_prediction(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['Income', 'CreditScore', 'DTIRatio', 'LoanAmount']]
    y = df['LoanApproved']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )


    nb_model = GaussianNB()


    nb_model.fit(X_train, y_train)


    y_pred = nb_model.predict(X_test)


    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)

    print("\nConfusion Matrix:")
    print(cm)
    print(f"\nMatrix Structure:")
    print(f"  [ True Negatives (TN): {cm[0][0]} | False Positives (FP): {cm[0][1]} ]")
    print(f"  [ False Negatives (FN): {cm[1][0]} | True Positives (TP):  {cm[1][1]} ]")

    print(f"\nAccuracy Score: {acc * 100:.2f}%")

    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("bank_loan_dataset.csv")
    run_bank_loan_prediction(dataset_file)

Dataset successfully created and saved as 'bank_loan_dataset.csv'.

--- Dataset Preview ---
   Income  CreditScore  DTIRatio  LoanAmount  LoanApproved
0     122          405      10.8          12             1
1     112          419      10.8          38             1
2      34          357      11.8          39             0
3     126          645      11.2          36             1
4      91          773      33.8          28             1 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[25  5]
 [ 0 30]]

Matrix Structure:
  [ True Negatives (TN): 25 | False Positives (FP): 5 ]
  [ False Negatives (FN): 0 | True Positives (TP):  30 ]

Accuracy Score: 91.67%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.83      0.91        30
           1       0.86      1.00      0.92        30

    accuracy                      

In [1]:
# Session 9: K-Nearest Neighbors (K-NN) Classification
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

def generate_dataset(filename="social_network_ads.csv"):
    np.random.seed(42)
    n_samples = 250

    age = np.random.randint(18, 60, size=n_samples)
    estimated_salary = np.random.randint(15000, 150000, size=n_samples)

    # Probability of purchasing increases with age and salary
    purchase_prob = 1 / (1 + np.exp(-(age * 0.12 + estimated_salary * 0.00003 - 7)))
    purchased = (np.random.rand(n_samples) < purchase_prob).astype(int)

    df = pd.DataFrame({
        'Age': age,
        'EstimatedSalary': estimated_salary,
        'Purchased': purchased
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_knn_classification(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['Age', 'EstimatedSalary']]
    y = df['Purchased']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # Standardize features (highly recommended for distance-based models like KNN)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and fit KNN Classifier (k=5)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, y_train)

    # Predictions
    y_pred = knn.predict(X_test_scaled)

    # Evaluation Metrics
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print("\nConfusion Matrix:")
    print(cm)
    print(f"\nAccuracy Score: {acc * 100:.2f}%")
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("social_network_ads.csv")
    run_knn_classification(dataset_file)

Dataset successfully created and saved as 'social_network_ads.csv'.

--- Dataset Preview ---
   Age  EstimatedSalary  Purchased
0   56            30151          0
1   46            81690          1
2   32            19499          0
3   25            21295          0
4   38           135885          0 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[28  6]
 [ 5 24]]

Accuracy Score: 82.54%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.82      0.84        34
           1       0.80      0.83      0.81        29

    accuracy                           0.83        63
   macro avg       0.82      0.83      0.82        63
weighted avg       0.83      0.83      0.83        63



In [2]:
# Session 10: Logistic Regression (LR) Algorithm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

def generate_dataset(filename="user_behavior.csv"):
    np.random.seed(42)
    n_samples = 300

    age = np.random.randint(18, 60, size=n_samples)
    time_spent_on_app = np.random.uniform(5.0, 120.0, size=n_samples) # in minutes

    # Probability of making an in-app purchase increases with time spent and age
    logit = 0.05 * time_spent_on_app + 0.08 * age - 6.0
    prob = 1 / (1 + np.exp(-logit))
    purchased = (np.random.rand(n_samples) < prob).astype(int)

    df = pd.DataFrame({
        'Age': age,
        'TimeSpent': time_spent_on_app,
        'Purchased': purchased
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_logistic_regression(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['Age', 'TimeSpent']]
    y = df['Purchased']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and train Logistic Regression
    lr_model = LogisticRegression(random_state=42)
    lr_model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = lr_model.predict(X_test_scaled)

    # Metrics
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print("\nConfusion Matrix:")
    print(cm)
    print(f"\nAccuracy Score: {acc * 100:.2f}%")
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("user_behavior.csv")
    run_logistic_regression(dataset_file)

Dataset successfully created and saved as 'user_behavior.csv'.

--- Dataset Preview ---
   Age   TimeSpent  Purchased
0   56   81.926267          1
1   46  104.379270          1
2   32   31.471306          0
3   25   62.407239          0
4   38   70.780483          1 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[18  9]
 [ 2 31]]

Accuracy Score: 81.67%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.67      0.77        27
           1       0.78      0.94      0.85        33

    accuracy                           0.82        60
   macro avg       0.84      0.80      0.81        60
weighted avg       0.83      0.82      0.81        60



In [3]:
# Session 11: Linear Regression (LR) Algorithm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def generate_dataset(filename="salary_data.csv"):
    np.random.seed(42)
    n_samples = 150

    years_experience = np.random.uniform(0.5, 15.0, size=n_samples).round(1)

    # Salary increases linearly with years of experience
    base_salary = 30000 + years_experience * 8500
    noise = np.random.normal(0, 7500, size=n_samples)
    salary = (base_salary + noise).round(2)

    df = pd.DataFrame({
        'YearsExperience': years_experience,
        'Salary': salary
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_linear_regression(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['YearsExperience']]
    y = df['Salary']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # Initialize and train Linear Regression model
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    # Predict
    y_pred = lr_model.predict(X_test)

    # Evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print(f"Mean Absolute Error (MAE) : ${mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R² Score (Accuracy)       : {r2 * 100:.2f}%")

    print("\n--- Model Parameters ---")
    print(f"  Coefficient (Slope)     : {lr_model.coef_[0]:,.2f}")
    print(f"  Intercept               : ${lr_model.intercept_:,.2f}")

    # Sample prediction
    sample_exp = pd.DataFrame({'YearsExperience': [5.5]})
    predicted_val = lr_model.predict(sample_exp)[0]
    print("\n--- Sample Prediction ---")
    print(f"Years of Experience: 5.5 years")
    print(f"Predicted Salary   : ${predicted_val:,.2f}")

if __name__ == "__main__":
    dataset_file = generate_dataset("salary_data.csv")
    run_linear_regression(dataset_file)

Dataset successfully created and saved as 'salary_data.csv'.

--- Dataset Preview ---
   YearsExperience     Salary
0              5.9   78707.29
1             14.3  153811.61
2             11.1  124089.66
3              9.2   99434.91
4              2.8   62371.17 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------
Mean Absolute Error (MAE) : $4,413.16
Root Mean Squared Error (RMSE): $5,171.41
R² Score (Accuracy)       : 98.15%

--- Model Parameters ---
  Coefficient (Slope)     : 8,649.45
  Intercept               : $29,653.76

--- Sample Prediction ---
Years of Experience: 5.5 years
Predicted Salary   : $77,225.74


In [4]:
# Session 12: Comparison of Linear and Polynomial Regression
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

def generate_nonlinear_dataset(filename="nonlinear_growth.csv"):
    np.random.seed(42)
    n_samples = 150

    # Feature: Temperature (°C)
    temp = np.random.uniform(10.0, 40.0, size=n_samples)

    # Target: Yield (has a quadratic relationship with temperature)
    base_yield = -0.5 * (temp - 25)**2 + 150
    noise = np.random.normal(0, 8, size=n_samples)
    yield_crop = (base_yield + noise).round(2)

    df = pd.DataFrame({
        'Temperature': temp,
        'Yield': yield_crop
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def compare_linear_vs_polynomial(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['Temperature']]
    y = df['Yield']

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # --- 1. Linear Regression ---
    lin_reg = LinearRegression()
    lin_reg.fit(X_train, y_train)
    y_pred_lin = lin_reg.predict(X_test)

    # --- 2. Polynomial Regression (Degree = 2) ---
    poly_features = PolynomialFeatures(degree=2)
    X_train_poly = poly_features.fit_transform(X_train)
    X_test_poly = poly_features.transform(X_test)

    poly_reg = LinearRegression()
    poly_reg.fit(X_train_poly, y_train)
    y_pred_poly = poly_reg.predict(X_test_poly)

    # --- Evaluation ---
    rmse_lin = np.sqrt(mean_squared_error(y_test, y_pred_lin))
    r2_lin = r2_score(y_test, y_pred_lin)

    rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
    r2_poly = r2_score(y_test, y_pred_poly)

    print("-" * 60)
    print("--- PERFORMANCE COMPARISON ---")
    print("-" * 60)
    print(f"Linear Regression:     RMSE = {rmse_lin:.2f}, R² Score = {r2_lin*100:.2f}%")
    print(f"Polynomial Regression: RMSE = {rmse_poly:.2f}, R² Score = {r2_poly*100:.2f}%")
    print("-" * 60)

    if r2_poly > r2_lin:
        print("Conclusion: Polynomial Regression successfully captured the non-linear trend and significantly outperformed the baseline Linear model!")
    else:
        print("Conclusion: Linear Regression is sufficient for this trend.")

if __name__ == "__main__":
    dataset_file = generate_nonlinear_dataset("nonlinear_growth.csv")
    compare_linear_vs_polynomial(dataset_file)

Dataset successfully created and saved as 'nonlinear_growth.csv'.

--- Dataset Preview ---
   Temperature   Yield
0    21.236204  141.38
1    38.521429   61.00
2    31.959818  125.50
3    27.959755  136.27
4    14.680559  105.90 

------------------------------------------------------------
--- PERFORMANCE COMPARISON ---
------------------------------------------------------------
Linear Regression:     RMSE = 32.83, R² Score = -10.59%
Polynomial Regression: RMSE = 5.51, R² Score = 96.89%
------------------------------------------------------------
Conclusion: Polynomial Regression successfully captured the non-linear trend and significantly outperformed the baseline Linear model!


In [5]:
# Session 13: Expectation-Maximization (EM) Algorithm
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

def generate_em_dataset(filename="multimodal_data.csv"):
    np.random.seed(42)
    n_samples = 150

    # Generate data from two distinct normal distributions (representing two clusters/classes)
    cluster1 = np.random.normal(loc=20.0, scale=4.0, size=(75, 2))
    cluster2 = np.random.normal(loc=35.0, scale=5.0, size=(75, 2))

    combined_data = np.vstack((cluster1, cluster2))

    df = pd.DataFrame(combined_data, columns=['Feature_1', 'Feature_2'])
    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_em_clustering(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)

    # Initialize and fit Gaussian Mixture Model (EM)
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm.fit(X_scaled)

    # Get cluster assignments and probabilities
    labels = gmm.predict(X_scaled)
    probs = gmm.predict_proba(X_scaled)

    df['Cluster'] = labels
    df['Prob_Cluster_0'] = probs[:, 0].round(4)
    df['Prob_Cluster_1'] = probs[:, 1].round(4)

    print("-" * 60)
    print("--- EM ALGORITHM CONVERGENCE & PARAMETERS ---")
    print("-" * 60)
    print(f"Converged                     : {gmm.converged_}")
    print(f"Number of iterations to converge: {gmm.n_iter_}")
    print("\nEstimated Means (Scaled):")
    print(gmm.means_)

    print("\n--- Sample Clustered Records ---")
    print(df.iloc[[0, 5, 75, 80]])

if __name__ == "__main__":
    dataset_file = generate_em_dataset("multimodal_data.csv")
    run_em_clustering(dataset_file)

Dataset successfully created and saved as 'multimodal_data.csv'.

--- Dataset Preview ---
   Feature_1  Feature_2
0  21.986857  19.446943
1  22.590754  26.092119
2  19.063387  19.063452
3  26.316851  23.069739
4  18.122102  22.170240 

------------------------------------------------------------
--- EM ALGORITHM CONVERGENCE & PARAMETERS ---
------------------------------------------------------------
Converged                     : True
Number of iterations to converge: 2

Estimated Means (Scaled):
[[-0.87620011 -0.84765898]
 [ 0.89800513  0.86875373]]

--- Sample Clustered Records ---
    Feature_1  Feature_2  Cluster  Prob_Cluster_0  Prob_Cluster_1
0   21.986857  19.446943        0             1.0             0.0
5   18.146329  18.137081        0             1.0             0.0
75  36.252464  36.732241        1             0.0             1.0
80  30.126592  38.935423        1             0.0             1.0


In [6]:
# Session 14: Credit Score Classification
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def generate_credit_dataset(filename="credit_score_data.csv"):
    np.random.seed(42)
    n_samples = 400

    # Generate typical financial indicators
    annual_income = np.random.randint(15000, 180000, size=n_samples)
    num_bank_accounts = np.random.randint(1, 10, size=n_samples)
    num_credit_cards = np.random.randint(1, 12, size=n_samples)
    interest_rate = np.random.randint(1, 35, size=n_samples)
    num_of_delayed_payment = np.random.randint(0, 25, size=n_samples)

    # Scoring function to determine credit score category
    score = (annual_income * 0.0002) - (num_bank_accounts * 1.5) - (num_credit_cards * 1.0) - (interest_rate * 0.8) - (num_of_delayed_payment * 2.0) + 50

    credit_score = []
    for s in score:
        if s > 45:
            credit_score.append('Good')
        elif s > 15:
            credit_score.append('Standard')
        else:
            credit_score.append('Poor')

    df = pd.DataFrame({
        'AnnualIncome': annual_income,
        'NumBankAccounts': num_bank_accounts,
        'NumCreditCards': num_credit_cards,
        'InterestRate': interest_rate,
        'NumDelayedPayments': num_of_delayed_payment,
        'CreditScore': credit_score
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_credit_classification(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")
    print("--- Class Distribution ---")
    print(df['CreditScore'].value_counts(), "\n")

    X = df.drop(columns=['CreditScore'])
    y = df['CreditScore']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    # Initialize and train Random Forest Classifier
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_classifier.fit(X_train, y_train)

    # Predict
    y_pred = rf_classifier.predict(X_test)

    # Evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=['Good', 'Standard', 'Poor'])

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print(f"Accuracy Score: {acc * 100:.2f}%")
    print("\nConfusion Matrix (Labels: Good, Standard, Poor):")
    print(cm)
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_credit_dataset("credit_score_data.csv")
    run_credit_classification(dataset_file)

Dataset successfully created and saved as 'credit_score_data.csv'.

--- Dataset Preview ---
   AnnualIncome  NumBankAccounts  NumCreditCards  InterestRate  \
0        136958                7               8             1   
1        161867                5               6            26   
2        146932                6               3            29   
3        118694                7               9            21   
4        134879                3               2            11   

   NumDelayedPayments CreditScore  
0                   4        Good  
1                  23        Poor  
2                   8    Standard  
3                   2    Standard  
4                   9    Standard   

--- Class Distribution ---
CreditScore
Poor        186
Standard    177
Good         37
Name: count, dtype: int64 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------
Accuracy Score: 89.00%

Confusion Matrix (Labels: G

In [7]:
# Session 15: Iris Flower Classification using KNN
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

def generate_dataset(filename="iris_knn_dataset.csv"):
    iris = load_iris()
    df = pd.DataFrame(data=iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
    df['species'] = iris.target_names[iris.target]
    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_iris_knn_classification(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
    y = df['species']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and fit KNN Classifier
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, y_train)

    # Predictions
    y_pred = knn.predict(X_test_scaled)

    # Evaluation metrics
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print("\nConfusion Matrix:")
    print(cm)
    print(f"\nAccuracy Score: {acc * 100:.2f}%")
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    dataset_file = generate_dataset("iris_knn_dataset.csv")
    run_iris_knn_classification(dataset_file)

Dataset successfully created and saved as 'iris_knn_dataset.csv'.

--- Dataset Preview ---
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------

Confusion Matrix:
[[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]

Accuracy Score: 93.33%

Detailed Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro av

In [8]:
# Session 16: Car Price Prediction Model using Python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def generate_car_dataset(filename="car_prices_dataset.csv"):
    np.random.seed(42)
    n_samples = 300

    # Generate typical car features
    brand = np.random.choice(['Toyota', 'Honda', 'Ford', 'BMW', 'Hyundai'], size=n_samples)
    year = np.random.randint(2010, 2023, size=n_samples)
    mileage = np.random.randint(5000, 150000, size=n_samples)
    engine_size = np.random.uniform(1.0, 4.0, size=n_samples).round(1)

    # Map brands to base multipliers
    brand_mult = {'BMW': 1.8, 'Toyota': 1.2, 'Honda': 1.1, 'Ford': 1.0, 'Hyundai': 0.9}
    base_multiplier = np.array([brand_mult[b] for b in brand])

    # Price model
    base_price = 25000 * base_multiplier
    depreciation = (2023 - year) * 1500 + (mileage * 0.08)
    engine_bonus = (engine_size - 1.0) * 4000

    price = base_price - depreciation + engine_bonus
    # Ensure minimum residual price
    price = np.clip(price, 2000, 85000)

    # Add noise
    noise = np.random.normal(0, 1500, size=n_samples)
    final_price = (price + noise).round(2)

    df = pd.DataFrame({
        'Brand': brand,
        'Year': year,
        'Mileage': mileage,
        'EngineSize': engine_size,
        'Price': final_price
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_car_price_prediction(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df.drop(columns=['Price'])
    y = df['Price']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # Preprocessor using ColumnTransformer to handle categorical text data
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(), ['Brand'])
        ], remainder='passthrough'
    )

    # Regression Pipeline with Random Forest
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ])

    # Train
    model_pipeline.fit(X_train, y_train)

    # Predict
    y_pred = model_pipeline.predict(X_test)

    # Evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print(f"Mean Absolute Error (MAE) : ${mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R² Score (Accuracy)       : {r2 * 100:.2f}%")

    # Sample prediction
    sample_car = pd.DataFrame({
        'Brand': ['BMW'],
        'Year': [2018],
        'Mileage': [50000],
        'EngineSize': [2.5]
    })
    pred_price = model_pipeline.predict(sample_car)[0]
    print("\n--- Sample Prediction ---")
    print(f"Features: Brand=BMW, Year=2018, Mileage=50,000 miles, Engine=2.5L")
    print(f"Predicted Car Price: ${pred_price:,.2f}")

if __name__ == "__main__":
    dataset_file = generate_car_dataset("car_prices_dataset.csv")
    run_car_price_prediction(dataset_file)

Dataset successfully created and saved as 'car_prices_dataset.csv'.

--- Dataset Preview ---
     Brand  Year  Mileage  EngineSize     Price
0      BMW  2014     6542         3.3  41238.27
1  Hyundai  2017    26677         1.6  12922.14
2     Ford  2022    44081         2.0  24922.13
3  Hyundai  2010    98105         2.3   3458.83
4  Hyundai  2014    24065         2.5  14007.51 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------
Mean Absolute Error (MAE) : $1,871.18
Root Mean Squared Error (RMSE): $2,347.78
R² Score (Accuracy)       : 94.01%

--- Sample Prediction ---
Features: Brand=BMW, Year=2018, Mileage=50,000 miles, Engine=2.5L
Predicted Car Price: $40,605.51


In [9]:
# Session 17: House Price Prediction using Random Forest Regressor
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def generate_house_dataset(filename="house_prices_rf.csv"):
    np.random.seed(42)
    n_samples = 300

    square_feet = np.random.randint(600, 4000, size=n_samples)
    bedrooms = np.random.randint(1, 6, size=n_samples)
    bathrooms = np.random.randint(1, 4, size=n_samples)
    age = np.random.randint(1, 40, size=n_samples)
    distance_to_center = np.random.uniform(0.5, 30.0, size=n_samples).round(1)

    # Price calculation has complex interactions (non-linearities suited for Random Forest)
    base_price = (
        square_feet * 165 +
        bedrooms * 12000 +
        bathrooms * 15000 -
        (age ** 1.2) * 800 -
        (distance_to_center ** 0.8) * 4000 +
        50000
    )

    noise = np.random.normal(0, 20000, size=n_samples)
    price = np.clip((base_price + noise).round(2), 50000, 950000)

    df = pd.DataFrame({
        'SquareFeet': square_feet,
        'Bedrooms': bedrooms,
        'Bathrooms': bathrooms,
        'Age': age,
        'DistanceToCenter': distance_to_center,
        'Price': price
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_house_price_rf_prediction(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")

    X = df[['SquareFeet', 'Bedrooms', 'Bathrooms', 'Age', 'DistanceToCenter']]
    y = df['Price']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # Initialize and fit Random Forest Regressor
    rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_regressor.fit(X_train, y_train)

    # Predictions
    y_pred = rf_regressor.predict(X_test)

    # Evaluation Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print(f"Mean Absolute Error (MAE) : ${mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R² Score (Accuracy)       : {r2 * 100:.2f}%")

    # Sample prediction
    sample_house = pd.DataFrame({
        'SquareFeet': [2200],
        'Bedrooms': [3],
        'Bathrooms': [2],
        'Age': [8],
        'DistanceToCenter': [6.5]
    })
    predicted_price = rf_regressor.predict(sample_house)[0]
    print("\n--- Sample Prediction ---")
    print(f"Features: 2200 sqft, 3 Bed, 2 Bath, 8 years old, 6.5 km from center")
    print(f"Predicted House Price: ${predicted_price:,.2f}")

if __name__ == "__main__":
    dataset_file = generate_house_dataset("house_prices_rf.csv")
    run_house_price_rf_prediction(dataset_file)

Dataset successfully created and saved as 'house_prices_rf.csv'.

--- Dataset Preview ---
   SquareFeet  Bedrooms  Bathrooms  Age  DistanceToCenter      Price
0        3774         2          3    9              17.2  688246.57
1        1460         3          1   26               2.4  296868.42
2        1894         4          3   22              23.4  397158.28
3        1730         2          1   30              13.9  277881.68
4        1695         1          1   17              16.0  303322.29 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------
Mean Absolute Error (MAE) : $22,471.77
Root Mean Squared Error (RMSE): $26,882.73
R² Score (Accuracy)       : 96.86%

--- Sample Prediction ---
Features: 2200 sqft, 3 Bed, 2 Bath, 8 years old, 6.5 km from center
Predicted House Price: $440,466.88


In [10]:
# Session 18: Compare Classification Algorithms and Evaluate Performance
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def generate_benchmark_dataset(filename="classification_benchmark.csv"):
    np.random.seed(42)
    n_samples = 400

    f1 = np.random.normal(loc=0.0, scale=1.0, size=n_samples)
    f2 = np.random.normal(loc=0.0, scale=1.0, size=n_samples)

    # Generate a non-linear decision boundary for 3 classes
    target = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        val = f1[i]**2 + f2[i]**2
        if val < 0.8:
            target[i] = 0 # Inner circle
        elif val < 2.2:
            target[i] = 1 # Middle ring
        else:
            target[i] = 2 # Outer region

    df = pd.DataFrame({
        'Feature_1': f1,
        'Feature_2': f2,
        'Class': target
    })
    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_classification_comparison(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")
    print("Class Counts:\n", df['Class'].value_counts(), "\n")

    X = df[['Feature_1', 'Feature_2']]
    y = df['Class']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Dictionary of classifiers
    models = {
        'Logistic Regression': LogisticRegression(multi_class='multinomial', random_state=42),
        'Naive Bayes': GaussianNB(),
        'Support Vector Machine (RBF)': SVC(kernel='rbf', random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
    }

    results = []

    print("-" * 80)
    print(f"{'Algorithm':<30} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
    print("-" * 80)

    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        print(f"{name:<30} | {acc*100:8.2f}% | {prec*100:9.2f}% | {rec*100:8.2f}% | {f1*100:8.2f}%")
        results.append({'Model': name, 'Accuracy': acc, 'F1-Score': f1})

    print("-" * 80)
    best_model = max(results, key=lambda x: x['Accuracy'])
    print(f"Best Performing Model: {best_model['Model']} ({best_model['Accuracy']*100:.2f}% Accuracy)")

if __name__ == "__main__":
    dataset_file = generate_benchmark_dataset("classification_benchmark.csv")
    run_classification_comparison(dataset_file)

Dataset successfully created and saved as 'classification_benchmark.csv'.

--- Dataset Preview ---
   Feature_1  Feature_2  Class
0   0.496714  -1.594428      2
1  -0.138264  -0.599375      0
2   0.647689   0.005244      0
3   1.523030   0.046981      2
4  -0.234153  -0.450065      0 

Class Counts:
 Class
0    141
2    130
1    129
Name: count, dtype: int64 

--------------------------------------------------------------------------------
Algorithm                      | Accuracy   | Precision  | Recall     | F1-Score  
--------------------------------------------------------------------------------
Logistic Regression            |    49.00% |     49.63% |    49.00% |    46.99%
Naive Bayes                    |    86.00% |     86.40% |    86.00% |    86.11%
Support Vector Machine (RBF)   |    95.00% |     95.15% |    95.00% |    95.03%


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Random Forest                  |    94.00% |     94.03% |    94.00% |    93.96%
--------------------------------------------------------------------------------
Best Performing Model: Support Vector Machine (RBF) (95.00% Accuracy)


In [11]:
# Session 19: Mobile Price Prediction using Random Forest Classifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def generate_mobile_dataset(filename="mobile_prices_dataset.csv"):
    np.random.seed(42)
    n_samples = 500

    # Generate typical mobile specifications
    battery_power = np.random.randint(500, 4000, size=n_samples) # mAh
    clock_speed = np.random.uniform(0.5, 3.2, size=n_samples).round(1) # GHz
    fc = np.random.randint(0, 20, size=n_samples) # Front Camera Megapixels
    int_memory = np.random.randint(2, 256, size=n_samples) # GB
    ram = np.random.randint(256, 8000, size=n_samples) # MB
    sc_h = np.random.randint(5, 20, size=n_samples) # Screen height

    # Calculate a custom scoring index to determine price range
    # RAM and Battery power have the highest impact on cost
    price_score = (ram * 0.45) + (battery_power * 0.15) + (int_memory * 0.8) + (clock_speed * 10) - (fc * 0.5)

    # Categorize into 4 price ranges (0: Low Cost, 1: Medium Cost, 2: High Cost, 3: Very High Cost)
    price_range = []
    for score in price_score:
        if score < 1200:
            price_range.append(0)
        elif score < 2400:
            price_range.append(1)
        elif score < 3600:
            price_range.append(2)
        else:
            price_range.append(3)

    df = pd.DataFrame({
        'BatteryPower': battery_power,
        'ClockSpeed': clock_speed,
        'FrontCamera': fc,
        'InternalMemory': int_memory,
        'RAM': ram,
        'ScreenHeight': sc_h,
        'PriceRange': price_range
    })

    df.to_csv(filename, index=False)
    print(f"Dataset successfully created and saved as '{filename}'.\n")
    return filename

def run_mobile_classification(csv_file):
    df = pd.read_csv(csv_file)
    print("--- Dataset Preview ---")
    print(df.head(), "\n")
    print("--- Price Range Counts ---")
    print(df['PriceRange'].value_counts().sort_index(), "\n")

    X = df.drop(columns=['PriceRange'])
    y = df['PriceRange']

    # Split dataset (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and train Random Forest Classifier
    rf_classifier = RandomForestClassifier(n_estimators=150, random_state=42)
    rf_classifier.fit(X_train_scaled, y_train)

    # Predictions
    y_pred = rf_classifier.predict(X_test_scaled)

    # Evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print("-" * 50)
    print("--- MODEL EVALUATION ---")
    print("-" * 50)
    print(f"Accuracy Score: {acc * 100:.2f}%")
    print("\nConfusion Matrix (Rows: Actual, Cols: Predicted):")
    print(cm)
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Low Cost', 'Medium Cost', 'High Cost', 'Very High Cost']))

    # Sample prediction
    sample_mobile = pd.DataFrame({
        'BatteryPower': [3000],
        'ClockSpeed': [2.4],
        'FrontCamera': [12],
        'InternalMemory': [128],
        'RAM': [4000], # 4GB
        'ScreenHeight': [15]
    })

    sample_scaled = scaler.transform(sample_mobile)
    pred_class = rf_classifier.predict(sample_scaled)[0]
    class_mapping = {0: 'Low Cost', 1: 'Medium Cost', 2: 'High Cost', 3: 'Very High Cost'}

    print("\n--- Sample Prediction ---")
    print("Features: Battery=3000mAh, Clock=2.4GHz, Front Cam=12MP, Int Memory=128GB, RAM=4000MB, Screen=15")
    print(f"Predicted Price Class: {class_mapping[pred_class]}")

if __name__ == "__main__":
    dataset_file = generate_mobile_dataset("mobile_prices_dataset.csv")
    run_mobile_classification(dataset_file)

Dataset successfully created and saved as 'mobile_prices_dataset.csv'.

--- Dataset Preview ---
   BatteryPower  ClockSpeed  FrontCamera  InternalMemory   RAM  ScreenHeight  \
0          3674         0.6           12               8   558             6   
1          1360         2.7            7              30  7143            17   
2          1794         1.5           17             140  4208            19   
3          1630         0.8            0              34  4939            12   
4          1595         1.9           14             135  1242            19   

   PriceRange  
0           0  
1           2  
2           1  
3           2  
4           0   

--- Price Range Counts ---
PriceRange
0     84
1    171
2    172
3     73
Name: count, dtype: int64 

--------------------------------------------------
--- MODEL EVALUATION ---
--------------------------------------------------
Accuracy Score: 91.00%

Confusion Matrix (Rows: Actual, Cols: Predicted):
[[17  0  0  0]
 [ 1 32

In [12]:
# Session 20: Single Layer Perceptron for Iris Classification
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

class SingleLayerPerceptron:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Initialize weights and bias to zeros
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        # Training loop
        for epoch in range(self.epochs):
            for idx, x_i in enumerate(X):
                # Calculate linear combination
                linear_output = np.dot(x_i, self.weights) + self.bias
                # Apply Heaviside step activation function
                y_predicted = 1 if linear_output >= 0.0 else 0

                # Perceptron Weight Update Rule: W = W + lr * (y - y_predicted) * X
                update = self.lr * (y[idx] - y_predicted)
                self.weights += update * x_i
                self.bias += update

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return np.where(linear_output >= 0.0, 1, 0)

def run_perceptron_iris():
    # Load standard Iris dataset
    iris = load_iris()
    X = iris.data[:, [2, 3]] # Use Petal Length and Petal Width as key features
    y = iris.target

    # Transform into a Binary Classification problem: Setosa (1) vs Non-Setosa (0)
    # This ensures the target classes are linearly separable
    y_binary = np.where(y == 0, 1, 0)
    class_names = ['Non-Setosa', 'Setosa']

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary, test_size=0.30, random_state=42, stratify=y_binary
    )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and train the Custom Perceptron
    perceptron = SingleLayerPerceptron(learning_rate=0.1, epochs=50)
    perceptron.fit(X_train_scaled, y_train)

    # Predict
    y_pred = perceptron.predict(X_test_scaled)

    # Evaluate metrics
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print("-" * 50)
    print("--- PERCEPTRON EVALUATION (Iris Setosa vs Non-Setosa) ---")
    print("-" * 50)
    print(f"Accuracy Score: {acc * 100:.2f}%")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    print("--- Model Parameters ---")
    print(f"Final Feature Weights: {perceptron.weights}")
    print(f"Final Bias: {perceptron.bias:.4f}")

if __name__ == "__main__":
    run_perceptron_iris()

--------------------------------------------------
--- PERCEPTRON EVALUATION (Iris Setosa vs Non-Setosa) ---
--------------------------------------------------
Accuracy Score: 100.00%

Confusion Matrix:
[[30  0]
 [ 0 15]]

Classification Report:
              precision    recall  f1-score   support

  Non-Setosa       1.00      1.00      1.00        30
      Setosa       1.00      1.00      1.00        15

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45

--- Model Parameters ---
Final Feature Weights: [-0.14910041 -0.07098754]
Final Bias: -0.1000
